# ✅ Solutions — Session 5: Time Series & I/O

Worked answers to every exercise plus the **Daily Sales Time Series** mini project.
Each solution is self-contained and uses pandas 3.x idioms only.

In [1]:
import io
import os

import numpy as np
import pandas as pd

## Exercise 1 — Parse dates and extract the year

In [2]:
dates = pd.to_datetime(pd.Series(["2024-01-01", "2024-02-15", "2024-03-20"]))
print("dtype:", dates.dtype)
dates.dt.year

dtype: datetime64[us]


0    2024
1    2024
2    2024
dtype: int32

## Exercise 2 — Count entries per weekday name

In [3]:
calendar = pd.Series(pd.date_range("2024-01-01", periods=30, freq="D"))
calendar.dt.day_name().value_counts()

Monday       5
Tuesday      5
Wednesday    4
Thursday     4
Friday       4
Saturday     4
Sunday       4
Name: count, dtype: int64

## Exercise 3 — Weekly totals and monthly means

In [4]:
rng = np.random.default_rng(0)
series_30 = pd.Series(
    rng.integers(100, 400, size=30),
    index=pd.date_range("2024-01-01", periods=30, freq="D"),
    name="sales",
)
series_30.head()

2024-01-01    355
2024-01-02    291
2024-01-03    253
2024-01-04    180
2024-01-05    192
Freq: D, Name: sales, dtype: int64

In [5]:
weekly_total = series_30.resample("W").sum()
monthly_mean = series_30.resample("ME").mean()

print("Weekly totals:")
print(weekly_total)
print("\nMonthly mean:")
print(monthly_mean)

Weekly totals:
2024-01-07    1505
2024-01-14    1798
2024-01-21    2091
2024-01-28    1696
2024-02-04     647
Freq: W-SUN, Name: sales, dtype: int64

Monthly mean:
2024-01-31    257.9
Freq: ME, Name: sales, dtype: float64


## Exercise 4 — Rolling mean and day-over-day change

The first `window - 1` values of a rolling result are `NaN` because a full
7-day window does not exist yet for the first six rows.

In [6]:
analysis = pd.DataFrame({
    "sales": series_30,
    "rolling_7": series_30.rolling(7).mean(),
    "pct_change": series_30.pct_change().round(4),
})
analysis.head(10)

,sales,rolling_7,pct_change
2024-01-01,355,NaN,NaN
2024-01-02,291,NaN,-0.1803
2024-01-03,253,NaN,-0.1306
2024-01-04,180,NaN,-0.2885
2024-01-05,192,NaN,0.0667
2024-01-06,112,NaN,-0.4167
2024-01-07,122,215.000000,0.0893
2024-01-08,104,179.142857,-0.1475
2024-01-09,152,159.285714,0.4615
2024-01-10,343,172.142857,1.2566


In [7]:
print("Missing rolling values in the first 10 rows:",
      analysis["rolling_7"].head(10).isna().sum())

Missing rolling values in the first 10 rows: 6


## Exercise 5 — Conceptual: choosing a file format

**Large daily sales for a pipeline → Parquet.** Parquet is a columnar, compressed
binary format that preserves dtypes exactly, reads only the columns you request, and
is far faster and smaller than text formats for large tables. The trade-off is that it
is not human-readable.

**Small preview for an Excel-only colleague → Excel (`.xlsx`).** Excel opens natively on
a colleague's machine, supports multiple named sheets, and needs no extra tooling. It is
not suited to large data or timezone-aware timestamps, and dates must be parsed again on
reload.

**CSV** sits in between: universally readable and easy to diff, but untyped (dtypes are
inferred on read) and comparatively large. The demonstration below shows the dtype
preservation difference.

In [8]:
sample = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=3, freq="D"),
    "sales": [10, 20, 30],
})

csv_text = sample.to_csv(index=False)
csv_inferred = pd.read_csv(io.StringIO(csv_text))
csv_parsed = pd.read_csv(io.StringIO(csv_text), parse_dates=["date"])

sample.to_parquet("format_demo.parquet", index=False)
parquet_back = pd.read_parquet("format_demo.parquet")
os.remove("format_demo.parquet")

print("CSV without parse_dates:", csv_inferred["date"].dtype)
print("CSV with parse_dates:   ", csv_parsed["date"].dtype)
print("Parquet:                ", parquet_back["date"].dtype)

CSV without parse_dates: str
CSV with parse_dates:    datetime64[us]
Parquet:                 datetime64[us]


## 🚀 Mini Project — Daily Sales Time Series

### Step 1 — Generate a year of daily sales

In [9]:
rng = np.random.default_rng(7)
dates = pd.date_range("2024-01-01", periods=365, freq="D")
trend = np.linspace(200, 400, 365)
weekly = 60 * np.sin(np.arange(365) * 2 * np.pi / 7)
noise = rng.normal(0, 25, 365)

sales = pd.DataFrame({"date": dates, "sales": (trend + weekly + noise).round(0)})
sales.head()

,date,sales
0,2024-01-01,200.0
1,2024-01-02,255.0
2,2024-01-03,253.0
3,2024-01-04,205.0
4,2024-01-05,165.0


### Step 2 — Set the index and compute monthly totals

In [10]:
ts = sales.set_index("date").sort_index()
monthly = ts["sales"].resample("ME").sum()
monthly

date
2024-01-31     6213.0
2024-02-29     6575.0
2024-03-31     7295.0
2024-04-30     7714.0
2024-05-31     8396.0
2024-06-30     8435.0
2024-07-31     9833.0
2024-08-31     9785.0
2024-09-30    10083.0
2024-10-31    11303.0
2024-11-30    11148.0
2024-12-31    11823.0
Freq: ME, Name: sales, dtype: float64

### Step 3 — Add a 7-day rolling mean and day-over-day growth

In [11]:
ts["rolling_7"] = ts["sales"].rolling(7).mean()
ts["pct_change"] = ts["sales"].pct_change().round(4)
ts[["sales", "rolling_7", "pct_change"]].head(10)

,sales,rolling_7,pct_change
date,,,
2024-01-01,200.0,NaN,NaN
2024-01-02,255.0,NaN,0.2750
2024-01-03,253.0,NaN,-0.0078
2024-01-04,205.0,NaN,-0.1897
2024-01-05,165.0,NaN,-0.1951
2024-01-06,119.0,NaN,-0.2788
2024-01-07,158.0,193.571429,0.3277
2024-01-08,237.0,198.857143,0.5000
2024-01-09,239.0,196.571429,0.0084


### Step 4 — Best and worst sales days

In [12]:
print("Best day: ", ts["sales"].idxmax(), "->", ts["sales"].max())
print("Worst day:", ts["sales"].idxmin(), "->", ts["sales"].min())

Best day:  2024-12-04 00:00:00 -> 471.0
Worst day: 2024-01-27 00:00:00 -> 93.0


### Step 5 — Save in three formats, reload, verify, and clean up

In [13]:
csv_path, xlsx_path, parquet_path = "mini.csv", "mini.xlsx", "mini.parquet"
out = ts.reset_index()
out.to_csv(csv_path, index=False)
out.to_excel(xlsx_path, index=False, sheet_name="Sales")
out.to_parquet(parquet_path, index=False)

csv_back = pd.read_csv(csv_path, parse_dates=["date"])
xlsx_back = pd.read_excel(xlsx_path, sheet_name="Sales", parse_dates=["date"])
parquet_back = pd.read_parquet(parquet_path)

print("CSV rows:    ", len(csv_back))
print("Excel rows:  ", len(xlsx_back))
print("Parquet rows:", len(parquet_back))

os.remove(csv_path)
os.remove(xlsx_path)
os.remove(parquet_path)

CSV rows:     365
Excel rows:   365
Parquet rows: 365


### Step 6 (Bonus) — CSV without `parse_dates` returns strings

The date column comes back as text unless you explicitly ask pandas to parse it,
which is why `parse_dates=` (or `pd.to_datetime`) is essential after a CSV reload.

In [14]:
csv_text = ts.reset_index().to_csv(index=False)
no_parse = pd.read_csv(io.StringIO(csv_text))
print("CSV without parse_dates:", no_parse["date"].dtype)

CSV without parse_dates: str
